# M12d · SVD et analyse en composantes principales — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m12d_svd_acp/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m12d_svd_acp/03_notebook.ipynb`, créée par `python tools/start_chapter.py M12d`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M12d.1 à M12d.13, M12d.20, M12d.21 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M12d.26 | La SVD par les valeurs propres de AᵀA · `svd` | 🔨 | ★★ | 25 |
|  | M12d.27 | Approximation de rang faible et conditionnement · `low_rank_approximation, condition_number` | 🔨 | ★ | 20 |
|  | M12d.28 | La pseudo-inverse · `pinv` | 🔨 | ★ | 15 |
|  | M12d.29 | Les axes principaux d'une matrice de covariance, et la variance expliquée · `principal_axes, explained_variance_ratio` | 🔨 | ★★ | 25 |
|  | M12d.30 | `np.linalg.svd`, `np.linalg.cond` et `sklearn.decomposition.PCA` contre tes fonctions : les signes | 📦 | ★ | 15 |
| B | M12d.31 | Prédire l'allure des valeurs singulières d'une image, et le rang qui suffit | 🔮 | ★ | 10 |
|  | M12d.32 | Compresser une image du domaine public : l'erreur en fonction du rang | 🔬 | ★★ | 20 |
|  | M12d.33 | La SVD par AᵀA contre `np.linalg.svd` : ce qu'on perd quand κ est grand | 🔬 | ★★ | 20 |
|  | M12d.34 | La SVD en images : Vᵀ, Σ puis U appliqués au cercle unité, étape par étape | 🎨 | ★ | 20 |
| C | M12d.35 | Chasse à l'erreur : une ACP sans centrer, n et n − 1 mélangés, des axes lus en lignes au lieu de colonnes | 🐛 | ★ | 15 |
|  | M12d.36 | Un test : la reconstruction de rang plein redonne X, et l'erreur décroît avec k | 🛠️ | ★ | 15 |
|  | M12d.37 | Défi : compresser une image centrée en gardant 93 % de son énergie avec au plus 5 % des nombres | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : SVD à la main, SVD compacte, image du cercle unité, approximation de rang faible et son erreur, coût de stockage, pseudo-inverse, conditionnement, axes principaux, scores et reconstruction, variance expliquée, ACP par la SVD et en grande dimension, voisinages de t-SNE, anomalies, changement d'unités.
- Écrire les six fonctions de `mymath/svd_pca.py` et les tester, puis comparer avec NumPy et scikit-learn.
- Prédire et mesurer la compression d'une photo, mesurer la précision de la SVD par AᵀA, dessiner la SVD, chasser des erreurs dans une ACP, écrire un test de propriétés, relever un défi de compression.

**Rappel express.** $\mathbf{A} = \mathbf{U}\mathbf{S}\mathbf{V}^\top = \sum_i s_i\mathbf{u}_i\mathbf{v}_i^\top$, $s_i = \sqrt{\lambda_i(\mathbf{A}^\top\mathbf{A})}$, $\mathbf{u}_i = \mathbf{A}\mathbf{v}_i/s_i$ ; $\lVert \mathbf{A}\mathbf{x} \rVert \le s_1\lVert \mathbf{x} \rVert$ ; $\lVert \mathbf{A} - \mathbf{A}_k \rVert_F^2 = \sum_{i > k} s_i^2$, $k(m + n + 1)$ nombres ; $\mathbf{A}^+ = \mathbf{V}\mathbf{S}^{-1}\mathbf{U}^\top$, $\mathbf{x}^+ = \mathbf{A}^+\mathbf{b}$ ; $\kappa = s_{\max}/s_{\min}$, $\kappa(\mathbf{A}^\top\mathbf{A}) = \kappa^2$ ; $\mathbf{C} = \frac{1}{n-1}\mathbf{X}_c^\top\mathbf{X}_c$, axes = vecteurs propres, $\lambda_j = s_j^2/(n - 1)$ ; scores $\mathbf{Z} = \mathbf{X}_c\mathbf{W}_k$, $\hat{\mathbf{X}} = \mathbf{Z}\mathbf{W}_k^\top + \boldsymbol{\mu}$ ; erreur moyenne $\frac{n-1}{n}\sum_{j > k}\lambda_j$ ; perplexité $2^{H}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M12d.1 à M12d.13, 🔗 M12d.20 et M12d.21)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0, -2, 5]` pour un vecteur (les valeurs singulières et les variances par ordre décroissant), une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex M12d.1 — La SVD d'une matrice 2 × 2 à la main, par AᵀA**

In [ ]:
# Ex M12d.1: the values you found on paper
answer_M12d_1a = ...  # a) a matrix
answer_M12d_1b = ...  # b) a list of 2 numbers
answer_M12d_1c = ...  # c) a list of 2 numbers
answer_M12d_1d = ...  # d) a list of 2 numbers (4 decimals)
answer_M12d_1e = ...  # e) a list of 2 numbers (4 decimals)
answer_M12d_1f = ...  # f) a list of 2 numbers (4 decimals)
answer_M12d_1g = ...  # g) a number
answer_M12d_1h = ...  # h) a list of 2 numbers (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_1a, answer_M12d_1b, answer_M12d_1c, answer_M12d_1d, answer_M12d_1e, answer_M12d_1f, answer_M12d_1g, answer_M12d_1h]):
    wbm.check(f"M12d.1{letter}", answer)

**Ex M12d.2 — La SVD compacte d'une matrice 3 × 2 de rang 1**

In [ ]:
# Ex M12d.2: the values you found on paper
answer_M12d_2a = ...  # a) an integer
answer_M12d_2b = ...  # b) a matrix
answer_M12d_2c = ...  # c) a list of 2 numbers
answer_M12d_2d = ...  # d) a number (4 decimals)
answer_M12d_2e = ...  # e) a list of 2 numbers (4 decimals)
answer_M12d_2f = ...  # f) a list of 3 numbers (4 decimals)
answer_M12d_2g = ...  # g) a number
answer_M12d_2h = ...  # h) a list of 2 numbers (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_2a, answer_M12d_2b, answer_M12d_2c, answer_M12d_2d, answer_M12d_2e, answer_M12d_2f, answer_M12d_2g, answer_M12d_2h]):
    wbm.check(f"M12d.2{letter}", answer)

**Ex M12d.3 — Le cercle unité devient une ellipse : axes et demi-axes**

In [ ]:
# Ex M12d.3: the values you found on paper
answer_M12d_3a = ...  # a) a list of 2 numbers
answer_M12d_3b = ...  # b) a list of 2 numbers (4 decimals)
answer_M12d_3c = ...  # c) a list of 2 numbers (4 decimals)
answer_M12d_3d = ...  # d) a number
answer_M12d_3e = ...  # e) a number (4 decimals)
answer_M12d_3f = ...  # f) a number (4 decimals)
answer_M12d_3g = ...  # g) a number (2 decimals)
answer_M12d_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12d_3a, answer_M12d_3b, answer_M12d_3c, answer_M12d_3d, answer_M12d_3e, answer_M12d_3f, answer_M12d_3g, answer_M12d_3h]):
    wbm.check(f"M12d.3{letter}", answer)

**Ex M12d.4 — L'approximation de rang 1 d'une matrice 3 × 3 et son erreur**

In [ ]:
# Ex M12d.4: the values you found on paper
answer_M12d_4a = ...  # a) a number
answer_M12d_4b = ...  # b) a matrix
answer_M12d_4c = ...  # c) a list of 3 numbers (4 decimals)
answer_M12d_4d = ...  # d) a list of 3 numbers (4 decimals)
answer_M12d_4e = ...  # e) a list of 3 numbers (4 decimals)
answer_M12d_4f = ...  # f) a matrix (4 decimals)
answer_M12d_4g = ...  # g) a number (4 decimals)
answer_M12d_4h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_4a, answer_M12d_4b, answer_M12d_4c, answer_M12d_4d, answer_M12d_4e, answer_M12d_4f, answer_M12d_4g, answer_M12d_4h]):
    wbm.check(f"M12d.4{letter}", answer)

**Ex M12d.5 — Compter les nombres stockés : une image en rang k**

In [ ]:
# Ex M12d.5: the values you found on paper
answer_M12d_5a = ...  # a) an integer
answer_M12d_5b = ...  # b) an integer
answer_M12d_5c = ...  # c) a number (4 decimals)
answer_M12d_5d = ...  # d) an integer
answer_M12d_5e = ...  # e) an integer
answer_M12d_5f = ...  # f) an integer
answer_M12d_5g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M12d_5a, answer_M12d_5b, answer_M12d_5c, answer_M12d_5d, answer_M12d_5e, answer_M12d_5f, answer_M12d_5g]):
    wbm.check(f"M12d.5{letter}", answer)

**Ex M12d.6 — La pseudo-inverse d'une matrice non inversible et la solution de norme minimale**

In [ ]:
# Ex M12d.6: the values you found on paper
answer_M12d_6a = ...  # a) an integer
answer_M12d_6b = ...  # b) a number (4 decimals)
answer_M12d_6c = ...  # c) a number
answer_M12d_6d = ...  # d) a matrix (6 decimals)
answer_M12d_6e = ...  # e) a list of 3 numbers (4 decimals)
answer_M12d_6f = ...  # f) a number
answer_M12d_6g = ...  # g) a number
answer_M12d_6h = ...  # h) a list of 2 numbers (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_6a, answer_M12d_6b, answer_M12d_6c, answer_M12d_6d, answer_M12d_6e, answer_M12d_6f, answer_M12d_6g, answer_M12d_6h]):
    wbm.check(f"M12d.6{letter}", answer)

**Ex M12d.7 — Le conditionnement de trois matrices, et l'erreur relative sur une solution**

In [ ]:
# Ex M12d.7: the values you found on paper
answer_M12d_7a = ...  # a) a number (1 decimal)
answer_M12d_7b = ...  # b) a number
answer_M12d_7c = ...  # c) a number (4 decimals)
answer_M12d_7d = ...  # d) a number (4 decimals)
answer_M12d_7e = ...  # e) a list of 2 numbers
answer_M12d_7f = ...  # f) a number (2 decimals)
answer_M12d_7g = ...  # g) a number (1 decimal)
answer_M12d_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12d_7a, answer_M12d_7b, answer_M12d_7c, answer_M12d_7d, answer_M12d_7e, answer_M12d_7f, answer_M12d_7g, answer_M12d_7h]):
    wbm.check(f"M12d.7{letter}", answer)

**Ex M12d.8 — Les axes principaux d'un nuage de ℝ³ : covariance, polynôme caractéristique, variances et axes**

In [ ]:
# Ex M12d.8: the values you found on paper
answer_M12d_8a = ...  # a) a list of 3 numbers
answer_M12d_8b = ...  # b) a matrix
answer_M12d_8c = ...  # c) a number
answer_M12d_8d = ...  # d) a number
answer_M12d_8e = ...  # e) a list of 3 numbers
answer_M12d_8f = ...  # f) a list of 3 numbers (4 decimals)
answer_M12d_8g = ...  # g) a list of 3 numbers (4 decimals)
answer_M12d_8h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M12d_8a, answer_M12d_8b, answer_M12d_8c, answer_M12d_8d, answer_M12d_8e, answer_M12d_8f, answer_M12d_8g, answer_M12d_8h]):
    wbm.check(f"M12d.8{letter}", answer)

**Ex M12d.9 — Scores, reconstruction et erreur de reconstruction moyenne avec deux composantes sur trois**

In [ ]:
# Ex M12d.9: the values you found on paper
answer_M12d_9a = ...  # a) a list of 2 numbers (4 decimals)
answer_M12d_9b = ...  # b) a list of 3 numbers (4 decimals)
answer_M12d_9c = ...  # c) a list of 7 numbers
answer_M12d_9d = ...  # d) a number
answer_M12d_9e = ...  # e) a number (4 decimals)
answer_M12d_9f = ...  # f) a number (4 decimals)
answer_M12d_9g = ...  # g) a number (4 decimals)
answer_M12d_9h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_9a, answer_M12d_9b, answer_M12d_9c, answer_M12d_9d, answer_M12d_9e, answer_M12d_9f, answer_M12d_9g, answer_M12d_9h]):
    wbm.check(f"M12d.9{letter}", answer)

**Ex M12d.10 — La variance expliquée cumulée : combien de composantes pour 95 % ?**

In [ ]:
# Ex M12d.10: the values you found on paper
answer_M12d_10a = ...  # a) a number
answer_M12d_10b = ...  # b) a number (4 decimals)
answer_M12d_10c = ...  # c) a list of 3 numbers (4 decimals)
answer_M12d_10d = ...  # d) an integer
answer_M12d_10e = ...  # e) an integer
answer_M12d_10f = ...  # f) a number (4 decimals)
answer_M12d_10g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M12d_10a, answer_M12d_10b, answer_M12d_10c, answer_M12d_10d, answer_M12d_10e, answer_M12d_10f, answer_M12d_10g]):
    wbm.check(f"M12d.10{letter}", answer)

**Ex M12d.11 — L'ACP par la SVD de la matrice centrée : σ² / (n − 1) donne les variances**

In [ ]:
# Ex M12d.11: the values you found on paper
answer_M12d_11a = ...  # a) a list of 3 numbers
answer_M12d_11b = ...  # b) a matrix
answer_M12d_11c = ...  # c) a list of 3 numbers
answer_M12d_11d = ...  # d) a list of 3 numbers (4 decimals)
answer_M12d_11e = ...  # e) a list of 3 numbers
answer_M12d_11f = ...  # f) a list of 3 numbers (4 decimals)
answer_M12d_11g = ...  # g) a list of 4 numbers (4 decimals)
answer_M12d_11h = ...  # h) a list of 4 numbers (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12d_11a, answer_M12d_11b, answer_M12d_11c, answer_M12d_11d, answer_M12d_11e, answer_M12d_11f, answer_M12d_11g, answer_M12d_11h]):
    wbm.check(f"M12d.11{letter}", answer)

**Ex M12d.12 — L'ACP quand n < p : passer par la petite matrice XXᵀ**

In [ ]:
# Ex M12d.12: the values you found on paper
answer_M12d_12a = ...  # a) a list of 5 numbers
answer_M12d_12b = ...  # b) a matrix
answer_M12d_12c = ...  # c) a list of 3 numbers
answer_M12d_12d = ...  # d) a list of 2 numbers
answer_M12d_12e = ...  # e) an integer
answer_M12d_12f = ...  # f) a list of 5 numbers (4 decimals)
answer_M12d_12g = ...  # g) a list of 3 numbers (4 decimals)

for letter, answer in zip("abcdefg", [answer_M12d_12a, answer_M12d_12b, answer_M12d_12c, answer_M12d_12d, answer_M12d_12e, answer_M12d_12f, answer_M12d_12g]):
    wbm.check(f"M12d.12{letter}", answer)

**Ex M12d.13 — Les probabilités de voisinage de t-SNE sur quatre points, et leur perplexité 2^H**

In [ ]:
# Ex M12d.13: the values you found on paper
answer_M12d_13a = ...  # a) a list of 3 numbers
answer_M12d_13b = ...  # b) a list of 3 numbers (4 decimals)
answer_M12d_13c = ...  # c) a number (4 decimals)
answer_M12d_13d = ...  # d) a number (4 decimals)
answer_M12d_13e = ...  # e) a number (4 decimals)
answer_M12d_13f = ...  # f) a number (4 decimals)
answer_M12d_13g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M12d_13a, answer_M12d_13b, answer_M12d_13c, answer_M12d_13d, answer_M12d_13e, answer_M12d_13f, answer_M12d_13g]):
    wbm.check(f"M12d.13{letter}", answer)

**Ex M12d.20 — Pont : détecter une anomalie par son erreur de reconstruction**

In [ ]:
# Ex M12d.20: the values you found on paper
answer_M12d_20a = ...  # a) a number
answer_M12d_20b = ...  # b) a number (4 decimals)
answer_M12d_20c = ...  # c) a number (4 decimals)
answer_M12d_20d = ...  # d) True or False
answer_M12d_20e = ...  # e) a number (4 decimals)
answer_M12d_20f = ...  # f) a number (4 decimals)
answer_M12d_20g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12d_20a, answer_M12d_20b, answer_M12d_20c, answer_M12d_20d, answer_M12d_20e, answer_M12d_20f, answer_M12d_20g]):
    wbm.check(f"M12d.20{letter}", answer)

**Ex M12d.21 — Pont : un changement d'unités diagonal D change Σ en DΣD, et les axes principaux avec ; la standardisation non**

In [ ]:
# Ex M12d.21: the values you found on paper
answer_M12d_21a = ...  # a) a matrix
answer_M12d_21b = ...  # b) a list of 3 numbers (4 decimals)
answer_M12d_21c = ...  # c) a list of 3 numbers (4 decimals)
answer_M12d_21d = ...  # d) a number (4 decimals)
answer_M12d_21e = ...  # e) a number (4 decimals)
answer_M12d_21f = ...  # f) a number (1 decimal)
answer_M12d_21g = ...  # g) a matrix (4 decimals)
answer_M12d_21h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12d_21a, answer_M12d_21b, answer_M12d_21c, answer_M12d_21d, answer_M12d_21e, answer_M12d_21f, answer_M12d_21g, answer_M12d_21h]):
    wbm.check(f"M12d.21{letter}", answer)

## Partie A · Ta librairie : SVD, rang faible, conditionnement, pseudo-inverse, axes principaux ; NumPy et scikit-learn

Fiche §1 à §7. Les six fonctions de `mymath/svd_pca.py`, puis les mêmes calculs avec NumPy et scikit-learn.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import StopCheck, checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def make_conditioned(kappa, seed=33, m=6, n=4):
    """A (m, n) matrix of singular values 1, ..., 1/kappa (geometric), from two random orthonormal bases."""
    rng = np.random.default_rng(seed)
    Q1, _ = np.linalg.qr(rng.normal(size=(m, n)))
    Q2, _ = np.linalg.qr(rng.normal(size=(n, n)))
    return (Q1 * kappa ** (-np.arange(n) / (n - 1))) @ Q2.T

### Ex M12d.26 — La SVD par les valeurs propres de AᵀA · `svd` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la SVD compacte par les valeurs et vecteurs propres de $\mathbf{A}^\top\mathbf{A}$, avec son seuil de rang numérique et sa convention de signe.  
**Prérequis :** fiche §1 · **mymath :** `svd_pca.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/svd_pca.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis, `@`, `.T`, `np.linalg.norm`, `np.linalg.solve` et `np.linalg.eigh` compris ; `np.linalg.svd`, `pinv`, `cond`, `lstsq`, `matrix_rank`, `scipy.linalg` et scikit-learn sont interdits (les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais la matrice reçue : travaille sur une copie, `np.array(A, dtype=float)`. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants.

Écris `svd(A, tol=1e-6)` (fiche §1), en suivant la docstring : une copie en flottants (`ValueError` si $\mathbf{A}$ n'est pas une matrice) ; `np.linalg.eigh` de $\mathbf{A}^\top\mathbf{A}$ ; les valeurs propres **écrêtées** à 0 (`np.clip`, avant la racine : un $-10^{-17}$ d'arrondi donnerait `nan`) et rangées par ordre décroissant, avec leurs vecteurs propres ; $s = \sqrt{\lambda}$ ; seules les valeurs **strictement** supérieures à `tol` fois la plus grande sont gardées ; la convention de signe sur chaque vecteur gardé (sa coordonnée de plus grande valeur absolue positive) ; enfin $\mathbf{U} = \mathbf{A}\mathbf{V}/s$, colonne par colonne. Une matrice nulle a $r = 0$ : `U`, `s` et `Vt` de formes $(m, 0)$, $(0,)$ et $(0, n)$.

La vérification calcule :
- a) les valeurs singulières de `svd([[2, 3, -1], [0, 4, 2]])` (une liste, 6 décimales) ;
- b) son `Vt` (une matrice, 6 décimales) ;
- c) vrai ou faux : pour une matrice $7 \times 4$ tirée au hasard (`np.random.default_rng(26)`, loi normale), `U @ np.diag(s) @ Vt` redonne $\mathbf{A}$, $\mathbf{U}^\top\mathbf{U}$ et $\mathbf{V}^\top\mathbf{V}$ valent $\mathbf{I}$, et `s` est égal aux valeurs de `np.linalg.svd`, le tout à $10^{-10}$ près ;
- d) le nombre de valeurs singulières gardées pour `[[1, -3], [2, -6], [-1, 3]]` ;
- e) les formes de `(U, s, Vt)` pour la matrice $3 \times 5$ `np.arange(15.0).reshape(3, 5) + np.eye(3, 5)`, en texte : `str((U.shape, s.shape, Vt.shape))` ;
- f) le nom de l'erreur levée par `svd([1, 2, 3])` ;
- g) vrai ou faux : après `svd(M)`, le tableau `M` (une copie en flottants de la matrice de a)) n'a pas changé.

Dans ta copie : retrouve a) à la main par $\mathbf{A}\mathbf{A}^\top$ (de forme $2 \times 2$, plus petite que $\mathbf{A}^\top\mathbf{A}$) : pourquoi a-t-elle les mêmes valeurs propres non nulles ? Pourquoi la matrice de d) n'a-t-elle qu'une valeur singulière, et que vaut-elle ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")  # reload your saved file

In [ ]:
A_26r = np.random.default_rng(26).normal(size=(7, 4))
with checked("M12d.26a"):
    wbm.check("M12d.26a", np.asarray(mymath.svd_pca.svd([[2, 3, -1], [0, 4, 2]])[1]).tolist(), computed=True)
with checked("M12d.26b"):
    wbm.check("M12d.26b", np.asarray(mymath.svd_pca.svd([[2, 3, -1], [0, 4, 2]])[2]).tolist(), computed=True)
with checked("M12d.26c"):
    U_26, s_26, Vt_26 = (np.asarray(x) for x in mymath.svd_pca.svd(A_26r))
    wbm.check("M12d.26c", bool(s_26.shape == (4,) and np.allclose(U_26 @ np.diag(s_26) @ Vt_26, A_26r, rtol=0, atol=1e-10)
                               and np.allclose(U_26.T @ U_26, np.eye(4), rtol=0, atol=1e-10)
                               and np.allclose(Vt_26 @ Vt_26.T, np.eye(4), rtol=0, atol=1e-10)
                               and np.allclose(s_26, np.linalg.svd(A_26r, compute_uv=False), rtol=0, atol=1e-10)),
              computed=True)
with checked("M12d.26d"):
    wbm.check("M12d.26d", len(mymath.svd_pca.svd([[1, -3], [2, -6], [-1, 3]])[1]), computed=True)
with checked("M12d.26e"):
    wbm.check("M12d.26e", str(tuple(np.asarray(x).shape for x in mymath.svd_pca.svd(np.arange(15.0).reshape(3, 5) + np.eye(3, 5)))),
              computed=True)
with checked("M12d.26f"):
    wbm.check("M12d.26f", raised(mymath.svd_pca.svd, [1, 2, 3]), computed=True)
with checked("M12d.26g"):
    M_26 = np.array([[2, 3, -1], [0, 4, 2]], dtype=float)
    mymath.svd_pca.svd(M_26)
    wbm.check("M12d.26g", bool(np.array_equal(M_26, np.array([[2, 3, -1], [0, 4, 2]], dtype=float))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12d_svd_pca.py", "test_svd_ or test_svd_pca_uses_")

### Ex M12d.27 — Approximation de rang faible et conditionnement · `low_rank_approximation, condition_number` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la meilleure approximation de rang $k$ et le conditionnement à partir de ta fonction `svd`.  
**Prérequis :** fiche §3 et §4 ; Ex M12d.26 · **mymath :** `svd_pca.py` · **Parcours :** C, E

Écris, en suivant les docstrings, avec ta fonction `svd` :
- `low_rank_approximation(A, k)` (fiche §3) : `ValueError` si `k` n'est pas un entier (un `bool` non plus) entre 0 et $\min(m, n)$ ; puis $\sum_{i \le k} s_i\mathbf{u}_i\mathbf{v}_i^\top$ (les $k$ premières colonnes de `U`, valeurs de `s`, lignes de `Vt` ; si `svd` en garde moins de $k$, toutes) ;
- `condition_number(A)` (fiche §4) : `ValueError` si $\mathbf{A}$ est nulle ; `math.inf` si `svd` garde moins de $\min(m, n)$ valeurs ; sinon $s_1/s_r$, en `float` de Python.

La vérification calcule :
- a) `low_rank_approximation([[4, 1, -2], [1, 3, 2]], 1)` (une matrice, 6 décimales) ;
- b) vrai ou faux : pour une matrice $6 \times 5$ tirée au hasard (`np.random.default_rng(27)`), $\lVert \mathbf{A} - \mathbf{A}_k \rVert_F^2 = \sum_{i > k} s_i^2$ à $10^{-9}$ près, relativement à $\lVert \mathbf{A} \rVert_F^2$, pour $k = 0, 1, \ldots, 5$ (les $s_i$ de `np.linalg.svd`) ;
- c) `condition_number([[5, 3], [4, 2.5]])` (4 décimales) ;
- d) vrai ou faux : `condition_number([[3, -6], [-1, 2]]) == math.inf` ;
- e) le nom de l'erreur levée par `low_rank_approximation([[1, 2], [3, 4]], 3)` ;
- f) le nom de l'erreur levée par `condition_number([[0, 0], [0, 0]])` ;
- g) vrai ou faux : `type(condition_number([[5, 3], [4, 2.5]]))` est `float`.

Dans ta copie : retrouve c) à la main, avec $s_1^2 + s_2^2 = \lVert \mathbf{A} \rVert_F^2$ et $s_1s_2 = \lvert\det\mathbf{A}\rvert$. Pourquoi la matrice de d) a-t-elle un conditionnement infini ? Que rend `low_rank_approximation(A, min(m, n))`, et pourquoi pas exactement $\mathbf{A}$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")  # reload your saved file

In [ ]:
A_27r = np.random.default_rng(27).normal(size=(6, 5))
s_27r = np.linalg.svd(A_27r, compute_uv=False)
with checked("M12d.27a"):
    wbm.check("M12d.27a", np.asarray(mymath.svd_pca.low_rank_approximation([[4, 1, -2], [1, 3, 2]], 1)).tolist(), computed=True)
with checked("M12d.27b"):
    errors_27 = [float(np.sum((A_27r - np.asarray(mymath.svd_pca.low_rank_approximation(A_27r, k))) ** 2)) for k in range(6)]
    wbm.check("M12d.27b", all(abs(e - float(np.sum(s_27r[k:] ** 2))) <= 1e-9 * float(np.sum(s_27r ** 2))
                              for k, e in enumerate(errors_27)), computed=True)
with checked("M12d.27c"):
    wbm.check("M12d.27c", mymath.svd_pca.condition_number([[5, 3], [4, 2.5]]), computed=True)
with checked("M12d.27d"):
    wbm.check("M12d.27d", bool(mymath.svd_pca.condition_number([[3, -6], [-1, 2]]) == math.inf), computed=True)
with checked("M12d.27e"):
    wbm.check("M12d.27e", raised(mymath.svd_pca.low_rank_approximation, [[1, 2], [3, 4]], 3), computed=True)
with checked("M12d.27f"):
    wbm.check("M12d.27f", raised(mymath.svd_pca.condition_number, [[0, 0], [0, 0]]), computed=True)
with checked("M12d.27g"):
    wbm.check("M12d.27g", type(mymath.svd_pca.condition_number([[5, 3], [4, 2.5]])) is float, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12d_svd_pca.py", "test_low_rank_ or test_condition_number_ or test_svd_pca_uses_")

### Ex M12d.28 — La pseudo-inverse · `pinv` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la pseudo-inverse par la SVD, avec son seuil sur les petites valeurs singulières.  
**Prérequis :** fiche §4 ; Ex M12d.26 · **mymath :** `svd_pca.py` · **Parcours :** C

Écris `pinv(A, rcond=1e-6)` (fiche §4), en suivant la docstring : `svd(A, tol=rcond)`, puis $\mathbf{V}\operatorname{diag}(1/s)\mathbf{U}^\top$ (divise les colonnes de `Vt.T` par `s`, sans former de matrice diagonale), de forme $(n, m)$ ; une matrice nulle donne des zéros.

La vérification calcule :
- a) `pinv([[2, 0, 1], [0, 3, 0]])` (une matrice, 6 décimales) ;
- b) vrai ou faux : pour une matrice $\mathbf{A}$ de forme $5 \times 3$ et de rang 2 tirée au hasard (cellule de vérification), $\mathbf{A}^+$ vérifie les quatre conditions de Penrose à $10^{-8}$ près : $\mathbf{A}\mathbf{A}^+\mathbf{A} = \mathbf{A}$, $\mathbf{A}^+\mathbf{A}\mathbf{A}^+ = \mathbf{A}^+$, $\mathbf{A}\mathbf{A}^+$ et $\mathbf{A}^+\mathbf{A}$ symétriques ;
- c) vrai ou faux : pour la même matrice et $\mathbf{b} = (1, 2, 3, 4, 5)$, `pinv(A) @ b` est la solution de `np.linalg.lstsq(A, b)` à $10^{-8}$ près ;
- d) la forme de `pinv(np.zeros((2, 3)))`, en texte : `str(pinv(...).shape)` ;
- e) l'élément `[1, 1]` de `pinv(np.diag([1.0, 1e-8]))`, puis de `pinv(np.diag([1.0, 1e-8]), rcond=1e-10)` (une liste de 2 nombres, 2 décimales) ;
- f) le nom de l'erreur levée par `pinv([1, 2])`.

Dans ta copie : retrouve a) à la main (les lignes de $\mathbf{A}$ sont orthogonales : $\mathbf{A}\mathbf{A}^\top$ est diagonale, et $\mathbf{A}^+ = \mathbf{A}^\top(\mathbf{A}\mathbf{A}^\top)^{-1}$). Que vaut $\mathbf{A}\mathbf{A}^+$ ? Pourquoi e) donne-t-il deux valeurs si différentes, et laquelle préférer si $10^{-8}$ est une erreur de mesure ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")  # reload your saved file

In [ ]:
rng_28 = np.random.default_rng(28)
A_28r = rng_28.normal(size=(5, 2)) @ rng_28.normal(size=(2, 3))
b_28 = np.arange(1.0, 6.0)
with checked("M12d.28a"):
    wbm.check("M12d.28a", np.asarray(mymath.svd_pca.pinv([[2, 0, 1], [0, 3, 0]])).tolist(), computed=True)
with checked("M12d.28b"):
    P_28 = np.asarray(mymath.svd_pca.pinv(A_28r))
    wbm.check("M12d.28b", bool(P_28.shape == (3, 5) and np.allclose(A_28r @ P_28 @ A_28r, A_28r, rtol=0, atol=1e-8)
                               and np.allclose(P_28 @ A_28r @ P_28, P_28, rtol=0, atol=1e-8)
                               and np.allclose(A_28r @ P_28, (A_28r @ P_28).T, rtol=0, atol=1e-8)
                               and np.allclose(P_28 @ A_28r, (P_28 @ A_28r).T, rtol=0, atol=1e-8)), computed=True)
with checked("M12d.28c"):
    wbm.check("M12d.28c", bool(np.allclose(np.asarray(mymath.svd_pca.pinv(A_28r)) @ b_28,
                                           np.linalg.lstsq(A_28r, b_28, rcond=None)[0], rtol=0, atol=1e-8)), computed=True)
with checked("M12d.28d"):
    wbm.check("M12d.28d", str(np.asarray(mymath.svd_pca.pinv(np.zeros((2, 3)))).shape), computed=True)
with checked("M12d.28e"):
    wbm.check("M12d.28e", [float(np.asarray(mymath.svd_pca.pinv(np.diag([1.0, 1e-8])))[1, 1]),
                           float(np.asarray(mymath.svd_pca.pinv(np.diag([1.0, 1e-8]), rcond=1e-10))[1, 1])], computed=True)
with checked("M12d.28f"):
    wbm.check("M12d.28f", raised(mymath.svd_pca.pinv, [1, 2]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12d_svd_pca.py", "test_pinv_ or test_svd_pca_uses_")

### Ex M12d.29 — Les axes principaux d'une matrice de covariance, et la variance expliquée · `principal_axes, explained_variance_ratio` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer les axes principaux d'une matrice de covariance, avec leurs contrôles et leur convention de signe, et la part de variance expliquée.  
**Prérequis :** fiche §5 à §7 · **mymath :** `svd_pca.py` · **Parcours :** C, E

Écris, en suivant les docstrings :
- `principal_axes(C, k=None)` (fiche §5) : `ValueError` si $\mathbf{C}$ n'est pas carrée, pas symétrique (relativement, comme dit la docstring), ou a une valeur propre sous $-10^{-12}\max(1, \lambda_{\max})$, ou si `k` n'est pas un entier entre 1 et $p$ ; puis `np.linalg.eigh`, l'ordre **décroissant**, les $k$ premiers axes **en colonnes** avec la convention de signe, et les variances (une valeur propre négative minuscule rendue comme 0, jamais $-0{,}0$) ;
- `explained_variance_ratio(variances)` (fiche §7) : `ValueError` si le tableau n'est pas 1-D, est vide, a une valeur négative ou une somme nulle ; sinon les parts, dans le même ordre.

La vérification calcule, avec $\mathbf{C}$ = `[[10, 0, -15], [0, 6, 0], [-15, 0, 26]]` :
- a) les axes rendus par `principal_axes(C)` (une matrice, 6 décimales) ;
- b) les variances rendues (une liste, 6 décimales) ;
- c) `explained_variance_ratio` de ces variances (une liste, 6 décimales) ;
- d) vrai ou faux : pour la covariance (`np.cov(..., rowvar=False)`) de $300$ exemples de $\mathbb{R}^5$ tirés au hasard (cellule de vérification), les axes sont orthonormés, vérifient $\mathbf{C}\mathbf{u}_j = \lambda_j\mathbf{u}_j$, et sont égaux aux lignes de `PCA().fit(X).components_` de scikit-learn, le tout à $10^{-8}$ près ;
- e) la forme des axes rendus par `principal_axes(C, k=1)`, en texte ;
- f) le nom de l'erreur levée par `principal_axes([[1, 2], [3, 4]])` ;
- g) le nom de l'erreur levée par `principal_axes([[1, 0], [0, -1]])` ;
- h) le nom de l'erreur levée par `explained_variance_ratio([0, 0])`.

Dans ta copie : retrouve a) et b) à la main, par la structure par blocs de $\mathbf{C}$ (fiche §5). Pourquoi la première colonne de a) a-t-elle une première coordonnée négative ? Que vaut la part cumulée des deux premiers axes ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")  # reload your saved file

In [ ]:
from sklearn.decomposition import PCA

X_29 = np.random.default_rng(29).normal(size=(300, 5)) @ np.diag([3.0, 2.0, 1.5, 1.0, 0.5])
cov_29 = np.cov(X_29, rowvar=False)
with checked("M12d.29a"):
    wbm.check("M12d.29a", np.asarray(mymath.svd_pca.principal_axes([[10, 0, -15], [0, 6, 0], [-15, 0, 26]])[0]).tolist(), computed=True)
with checked("M12d.29b"):
    wbm.check("M12d.29b", np.asarray(mymath.svd_pca.principal_axes([[10, 0, -15], [0, 6, 0], [-15, 0, 26]])[1]).tolist(), computed=True)
with checked("M12d.29c"):
    wbm.check("M12d.29c", np.asarray(mymath.svd_pca.explained_variance_ratio(
        mymath.svd_pca.principal_axes([[10, 0, -15], [0, 6, 0], [-15, 0, 26]])[1])).tolist(), computed=True)
with checked("M12d.29d"):
    axes_29, values_29 = (np.asarray(x) for x in mymath.svd_pca.principal_axes(cov_29))
    wbm.check("M12d.29d", bool(axes_29.shape == (5, 5) and np.allclose(axes_29.T @ axes_29, np.eye(5), rtol=0, atol=1e-8)
                               and np.allclose(cov_29 @ axes_29, axes_29 * values_29, rtol=0, atol=1e-8)
                               and np.allclose(axes_29.T, PCA().fit(X_29).components_, rtol=0, atol=1e-8)), computed=True)
with checked("M12d.29e"):
    wbm.check("M12d.29e", str(np.asarray(mymath.svd_pca.principal_axes([[10, 0, -15], [0, 6, 0], [-15, 0, 26]], k=1)[0]).shape), computed=True)
with checked("M12d.29f"):
    wbm.check("M12d.29f", raised(mymath.svd_pca.principal_axes, [[1, 2], [3, 4]]), computed=True)
with checked("M12d.29g"):
    wbm.check("M12d.29g", raised(mymath.svd_pca.principal_axes, [[1, 0], [0, -1]]), computed=True)
with checked("M12d.29h"):
    wbm.check("M12d.29h", raised(mymath.svd_pca.explained_variance_ratio, [0, 0]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12d_svd_pca.py", "test_principal_axes_ or test_explained_variance_ratio_ or test_svd_pca_uses_")

### Ex M12d.30 — `np.linalg.svd`, `np.linalg.cond` et `sklearn.decomposition.PCA` contre tes fonctions : les signes 📦 ★ ⏱️ 15 min
**Objectif :** utiliser `np.linalg.svd`, `np.linalg.cond` et `sklearn.decomposition.PCA`, lire ce qu'elles rendent, aligner leurs signes sur la convention, et comparer avec ta librairie.  
**Prérequis :** fiche §1, §4 et §6 ; Ex M12d.26 à M12d.29 · **Parcours :** C, E

En pratique, on ne réécrit pas ces fonctions :
- `np.linalg.svd(A, full_matrices=False)` rend `(U, s, Vt)`, les $\mathbf{v}_i$ en **lignes** de `Vt`, avec des signes **arbitraires** (ceux de LAPACK) ; `compute_uv=False` ne rend que `s` ;
- `np.linalg.cond(A)` rend $s_{\max}/s_{\min}$ ;
- `PCA().fit(X)` (scikit-learn) centre, fait la SVD de $\mathbf{X}_c$ (ou diagonalise sa covariance, selon la taille), et range les axes en **lignes** de `components_`, avec la convention de signe de la fiche ; `explained_variance_` donne les variances (en $n - 1$), `transform(X)` les scores.

Avec `M_30` et `X_30` (la cellule fournie), calcule :
- a) les valeurs singulières de `M_30` par `np.linalg.svd` (une liste, 6 décimales) ;
- b) vrai ou faux : après avoir multiplié chaque ligne de `Vt` (et la colonne de `U` qui va avec) par le signe qui rend positive sa coordonnée de plus grande valeur absolue, `U` et `Vt` de NumPy sont égaux à ceux de ta fonction `svd(M_30)`, à $10^{-8}$ près ;
- c) vrai ou faux : ta fonction `condition_number(M_30)` est égale à `np.linalg.cond(M_30)` à $10^{-10}$ près, relativement ;
- d) la forme de `np.cov(X_30)`, **sans** argument `rowvar` (une liste de 2 entiers) ;
- e) vrai ou faux : `PCA().fit(X_30).components_` est égal à la transposée des axes de ta fonction `principal_axes(np.cov(X_30, rowvar=False))`, à $10^{-8}$ près ;
- f) vrai ou faux : `PCA().fit(X_30).explained_variance_` est égal aux variances de ta fonction, à $10^{-8}$ près, relativement ;
- g) l'entier `d` tel que `np.var(scores, axis=0, ddof=d)` redonne `explained_variance_`, pour `scores = PCA().fit(X_30).transform(X_30)`.

Dans ta copie : pourquoi deux SVD justes peuvent-elles différer par des signes, et lesquels exactement (pourquoi faut-il changer **à la fois** $\mathbf{u}_i$ et $\mathbf{v}_i$) ? Que calcule `np.cov(X_30)` sans `rowvar=False`, et pourquoi est-ce un piège ? Pourquoi `explained_variance_` ne dépend-il pas de la moyenne $(10, -3, 5, 1)$ ajoutée aux données ?

In [ ]:
from sklearn.decomposition import PCA

M_30 = np.array([[3, -1, 2], [0, 4, -2], [1, 1, 5], [-2, 3, 0], [4, 0, 1]], dtype=float)
X_30 = (np.random.default_rng(30).normal(size=(200, 4))
        @ np.array([[2.0, 0.5, 0.0, 0.0], [0.0, 1.0, 0.8, 0.0], [0.0, 0.0, 0.6, 0.3], [0.4, 0.0, 0.0, 0.2]])
        + np.array([10.0, -3.0, 5.0, 1.0]))

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")
answer_M12d_30a = ...  # a) a list of 3 numbers (6 decimals)
answer_M12d_30d = ...  # d) a list of 2 integers
answer_M12d_30g = ...  # g) an integer
answer_M12d_30b = answer_M12d_30c = answer_M12d_30e = answer_M12d_30f = ...
# Each block stands alone: compute inside it what it needs, so that "Run all" goes on when a function is not written.
with wbm.attempt("M12d.30b"):        # your svd is called here: ⏳ while it is not written
    answer_M12d_30b = ...  # b) True or False
with wbm.attempt("M12d.30c"):        # your condition_number
    answer_M12d_30c = ...  # c) True or False
with wbm.attempt("M12d.30e"):        # your principal_axes
    answer_M12d_30e = ...  # e) True or False
with wbm.attempt("M12d.30f"):        # your principal_axes
    answer_M12d_30f = ...  # f) True or False

In [ ]:
for letter, answer in zip("abcdefg", [answer_M12d_30a, answer_M12d_30b, answer_M12d_30c, answer_M12d_30d,
                                       answer_M12d_30e, answer_M12d_30f, answer_M12d_30g]):
    wbm.check(f"M12d.30{letter}", answer)

## Partie B · Une prédiction, deux expériences et une figure

Fiche §1 à §4. Les valeurs singulières d'une photo, sa compression par le rang faible, la précision de la SVD par AᵀA, et la géométrie de la SVD.

### Ex M12d.31 — Prédire l'allure des valeurs singulières d'une image, et le rang qui suffit 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la décroissance des valeurs singulières d'une photo et d'un bruit, et le rang qui garde 90 % de leur énergie, puis le vérifier.  
**Prérequis :** fiche §3 · **Parcours :** C, E

La cellule fournie charge la photo de la tasse de café (`wbm.datasets.load_image("tasse")`, niveaux de gris de 0 à 255, licence CC0) et fabrique une image `NOISE_31` de **bruit pur**, de même taille, même moyenne et même écart-type (des niveaux tirés au hasard, indépendants). On retire à chaque image son niveau de gris moyen avant la SVD (fiche §3, ⚠️). Pour chaque question, choisis le numéro de la bonne réponse :
- a) l'allure des 400 valeurs singulières de la photo centrée, en échelle logarithmique : (1) quelques grandes valeurs, puis exactement zéro ; (2) une chute rapide, puis une longue traîne qui décroît lentement ; (3) toutes à peu près égales ;
- b) le plus petit rang qui garde au moins 90 % de l'énergie de la photo centrée : (1) de 1 à 4 ; (2) de 5 à 30 ; (3) de 31 à 150 ; (4) plus de 150 ;
- c) la même question pour le bruit : mêmes choix ;
- d) la part de l'énergie de la photo **non centrée** gardée au rang 1 : (1) moins de 30 % ; (2) entre 30 et 70 % ; (3) plus de 70 %.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

In [ ]:
A_31 = wbm.datasets.load_image("tasse")      # the coffee cup: 400 × 600 grey levels (data/cards/tasse_cafe.md)
NOISE_31 = np.random.default_rng(31).normal(loc=A_31.mean(), scale=A_31.std(), size=A_31.shape)   # pure noise
plt.figure(figsize=(5, 3.4))
plt.imshow(A_31, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.title("La tasse de café (400 × 600, CC0)")
plt.show()

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M12d_31a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M12d_31b = ...  # b) 1, 2, 3 or 4
prediction_M12d_31c = ...  # c) 1, 2, 3 or 4
prediction_M12d_31d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M12d_31a, prediction_M12d_31b, prediction_M12d_31c,
                                      prediction_M12d_31d]):
    wbm.check(f"M12d.31{letter}", prediction)

**L'expérience** (avec `np.linalg.svd`). Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M12d_31a, prediction_M12d_31b, prediction_M12d_31c, prediction_M12d_31d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    def rank_for(A, share):
        """The smallest rank k whose approximation keeps at least `share` of the energy of A − mean(A)."""
        s = np.linalg.svd(A - A.mean(), compute_uv=False)
        return int(np.searchsorted(np.cumsum(s ** 2) / np.sum(s ** 2), share) + 1)


    s_photo = np.linalg.svd(A_31 - A_31.mean(), compute_uv=False)
    s_noise = np.linalg.svd(NOISE_31 - NOISE_31.mean(), compute_uv=False)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.semilogy(np.arange(1, 401), s_photo, label="la photo, centrée")
    ax.semilogy(np.arange(1, 401), s_noise, label="du bruit de même taille, centré")
    ax.set_xlabel("rang i")
    ax.set_ylabel("valeur singulière s_i")
    ax.legend()
    plt.show()
    s_raw = np.linalg.svd(A_31, compute_uv=False)
    print(f"rang pour 90 % de l'énergie : photo {rank_for(A_31, 0.9)}, bruit {rank_for(NOISE_31, 0.9)}")
    print(f"part de l'énergie de la photo NON centrée gardée au rang 1 : {s_raw[0] ** 2 / np.sum(s_raw ** 2):.1%}".replace(".", ","))

**Après l'expérience**, dans ta copie : pourquoi la photo n'a-t-elle aucune valeur singulière nulle, alors que quelques couches suffisent à la reconnaître ? Pourquoi le bruit demande-t-il presque toutes ses couches, et quel lien avec le débruitage par troncature (fiche §3, en machine learning) ? Que dit d) sur le piège de l'image non centrée ?

### Ex M12d.32 — Compresser une image du domaine public : l'erreur en fonction du rang 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'énergie gardée, l'erreur et le coût de stockage d'une approximation de rang $k$ d'une photo, et regarder les reconstructions.  
**Prérequis :** fiche §3 ; Ex M12d.5 et M12d.31 · **Parcours :** C

Avec la photo `A_32` (la même qu'en M12d.31), écris :
- `reconstruct(A, k)` : l'image reconstruite au rang $k$ : on retire le niveau moyen $\bar{a}$, on prend l'approximation de rang $k$ de $\mathbf{A} - \bar{a}$ (`np.linalg.svd`, ou ta fonction `low_rank_approximation`), et l'on rajoute $\bar{a}$ ;
- `energy_kept(A)` : le tableau des parts d'énergie gardées aux rangs $1, 2, \ldots, \min(m, n)$, pour l'image **centrée** : $\sum_{i \le k} s_i^2 / \sum_i s_i^2$ (`np.cumsum`).

Puis calcule :
- a) le niveau de gris moyen $\bar{a}$ (2 décimales) ;
- b) la part d'énergie gardée au rang 10 (4 décimales) ;
- c) le plus petit rang qui garde au moins 90 % de l'énergie ;
- d) le plus petit rang qui garde au moins 99 % ;
- e) l'erreur relative $\lVert \mathbf{A} - \hat{\mathbf{A}}_{50} \rVert_F / \lVert \mathbf{A} - \bar{a} \rVert_F$, calculée **sur les matrices** avec `reconstruct` (4 décimales) ;
- f) la part des nombres stockés au rang de c), $k(m + n + 1)/(mn)$ (4 décimales) ;
- g) la part d'énergie gardée au rang 1 par l'image **non centrée** (sa SVD à elle, 4 décimales).

Une cellule affiche ensuite les reconstructions aux rangs 5, 20 et 50.

Dans ta copie : vérifie que e) est $\sqrt{1 - (\text{part gardée au rang } 50)}$, et dis pourquoi (Eckart-Young). À partir de quel rang la tasse est-elle reconnaissable ? Le grain du bois ? Combien de nombres faudrait-il pour garder 99 %, et est-ce encore une compression ? Pourquoi un JPEG fait-il beaucoup mieux (il travaille par petits blocs, et compte des bits) ?

In [ ]:
A_32 = wbm.datasets.load_image("tasse")      # 400 × 600 grey levels, CC0

In [ ]:
def reconstruct(A, k):
    """The rank-k approximation of A − mean(A), plus mean(A)."""
    raise NotImplementedError("np.linalg.svd(A - A.mean(), full_matrices=False), les k premières couches, puis + A.mean()")


def energy_kept(A):
    """The shares of the energy of A − mean(A) kept at ranks 1, 2, ..., min(m, n)."""
    raise NotImplementedError("np.linalg.svd(..., compute_uv=False), puis np.cumsum(s ** 2) / np.sum(s ** 2)")


answer_M12d_32a = ...  # a) a number (2 decimals)
answer_M12d_32b = answer_M12d_32c = answer_M12d_32d = answer_M12d_32e = answer_M12d_32f = answer_M12d_32g = ...
with wbm.attempt("M12d.32b"):        # your energy_kept is called here: ⏳ while it is not written
    answer_M12d_32b = ...  # b) a number (4 decimals)
    answer_M12d_32c = ...  # c) an integer
    answer_M12d_32d = ...  # d) an integer
with wbm.attempt("M12d.32e"):        # your reconstruct
    answer_M12d_32e = ...  # e) a number (4 decimals)
answer_M12d_32f = ...  # f) a number (4 decimals)
answer_M12d_32g = ...  # g) a number (4 decimals)

In [ ]:
for letter, answer in zip("abcdefg", [answer_M12d_32a, answer_M12d_32b, answer_M12d_32c, answer_M12d_32d,
                                       answer_M12d_32e, answer_M12d_32f, answer_M12d_32g]):
    wbm.check(f"M12d.32{letter}", answer)

with checked("M12d.32 figure"):
    fig, axes = plt.subplots(1, 4, figsize=(15, 3))
    try:
        for ax, k in zip(axes, (None, 5, 20, 50)):
            ax.imshow(A_32 if k is None else np.clip(reconstruct(A_32, k), 0, 255), cmap="gray", vmin=0, vmax=255)
            ax.set_title("l'original" if k is None else f"rang {k}")
            ax.axis("off")
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

### Ex M12d.33 — La SVD par AᵀA contre `np.linalg.svd` : ce qu'on perd quand κ est grand 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer la précision de la SVD par $\mathbf{A}^\top\mathbf{A}$ et de celle de LAPACK quand le conditionnement grandit, et voir ce qu'une bonne reconstruction cache.  
**Prérequis :** fiche §1 et §4 ; Ex M12d.26 · **Parcours :** C

La fonction `make_conditioned(kappa)` (cellule des outils) fabrique une matrice $6 \times 4$ dont les valeurs singulières, **connues**, vont de 1 à $1/\kappa$ (géométriquement) : son conditionnement est $\kappa$. Écris :
- `error_table()` : pour chaque $\kappa$ de `KAPPAS_33`, le couple des erreurs relatives sur la plus petite valeur singulière, $\lvert \hat{s}_{\min} - 1/\kappa \rvert \cdot \kappa$, obtenues par ta fonction `mymath.svd_pca.svd(A, tol=1e-12)` (pour qu'elle garde les quatre valeurs) et par `np.linalg.svd(A, compute_uv=False)`.

La cellule suivante affiche le tableau. Lis-le, puis réponds :
- a) le plus petit $\kappa$ de `KAPPAS_33` pour lequel l'erreur relative de **ta** fonction dépasse $10^{-6}$ ;
- b) vrai ou faux : l'erreur relative de `np.linalg.svd` reste sous $10^{-6}$ pour les quatre $\kappa$ ;
- c) vrai ou faux : pour $\kappa = 10^6$, ta fonction redonne quand même la matrice : $\lVert \mathbf{U}\operatorname{diag}(s)\mathbf{V}^\top - \mathbf{A} \rVert_F \le 10^{-12}\lVert \mathbf{A} \rVert_F$ (avec `tol=1e-12`) ;
- d) vrai ou faux : pour $\kappa = 10^6$, les colonnes du $\mathbf{U}$ de ta fonction sont orthonormées à $10^{-8}$ près (le plus grand écart de $\mathbf{U}^\top\mathbf{U}$ à $\mathbf{I}$) ;
- e) le nombre de valeurs singulières gardées par ta fonction **avec son seuil par défaut** pour $\kappa = 10^8$ ;
- f) `np.linalg.matrix_rank(make_conditioned(1e8))`.

Si ta fonction `svd` n'est pas encore écrite, la cellule utilise la référence et le dit.

Dans ta copie : de combien de puissances de 10 ton erreur grandit-elle quand $\kappa$ est multiplié par 100, et pourquoi (fiche §4, $\kappa^2$) ? Pourquoi c) reste-t-il vrai alors que $s_{\min}$ est faux : quelle part de $\mathbf{A}$ porte la couche fausse ? Que disent e) et f) du « rang » d'une matrice en flottants ?

In [ ]:
KAPPAS_33 = [1e2, 1e4, 1e6, 1e8]       # make_conditioned(kappa) (the tools): singular values 1, ..., 1/kappa

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12d")
try:                                             # your svd if it runs, else the reference
    mymath.svd_pca.svd(make_conditioned(1e2))
    svd_33 = mymath.svd_pca.svd
    print("avec ta fonction svd")
except Exception:  # noqa: BLE001 - not written yet (NotImplementedError), missing file, or a bug
    svd_33 = wbm.load_mymath("ref", alias="mymath_ref").svd_pca.svd
    print("ℹ️ ta fonction svd n'est pas encore écrite (ou lève une erreur) : l'expérience utilise la référence")


def error_table():
    """For each kappa of KAPPAS_33: (relative error on s_min of svd_33(A, tol=1e-12), the same for np.linalg.svd)."""
    raise NotImplementedError("une liste de 4 couples : |s_min − 1/κ| × κ pour svd_33 et pour np.linalg.svd")

In [ ]:
with checked("M12d.33 tableau"):
    table_33 = error_table()
    print("    κ      erreur relative sur s_min : ta svd    np.linalg.svd")
    for kappa, (mine, lapack) in zip(KAPPAS_33, table_33):
        print(f"{kappa:8.0e}   {mine:24.1e}   {lapack:14.1e}")

answer_M12d_33a = ...  # a) a number of KAPPAS_33
answer_M12d_33b = ...  # b) True or False
answer_M12d_33c = ...  # c) True or False, computed with svd_33
answer_M12d_33d = ...  # d) True or False, computed with svd_33
answer_M12d_33e = ...  # e) an integer
answer_M12d_33f = ...  # f) an integer
for letter, answer in zip("abcdef", [answer_M12d_33a, answer_M12d_33b, answer_M12d_33c, answer_M12d_33d,
                                     answer_M12d_33e, answer_M12d_33f]):
    wbm.check(f"M12d.33{letter}", answer)

### Ex M12d.34 — La SVD en images : Vᵀ, Σ puis U appliqués au cercle unité, étape par étape 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner les trois étapes de la SVD d'une matrice $2 \times 2$ sur le cercle unité et ses vecteurs singuliers.  
**Prérequis :** fiche §2 ; Ex M12d.3 · **Parcours :** C, E

Écris `draw_svd_steps(axes, A)` : avec `U, s, Vt = np.linalg.svd(A)` (ou ta fonction `svd`) et 200 points $\mathbf{x}$ du cercle unité, dessine dans les quatre axes reçus (sans en ouvrir d'autres) :
1. le cercle, et les vecteurs $\mathbf{v}_1$ et $\mathbf{v}_2$ (deux flèches) ;
2. l'image du cercle par $\mathbf{V}^\top$ (encore un cercle), et les images de $\mathbf{v}_1$ et $\mathbf{v}_2$ : $\mathbf{e}_1$ et $\mathbf{e}_2$ ;
3. l'image par $\mathbf{S}\mathbf{V}^\top$ : une ellipse alignée sur les axes, de demi-axes $s_1\mathbf{e}_1$ et $s_2\mathbf{e}_2$ ;
4. l'image par $\mathbf{U}\mathbf{S}\mathbf{V}^\top = \mathbf{A}$ : l'ellipse finale, et ses demi-axes $s_1\mathbf{u}_1$ et $s_2\mathbf{u}_2$.

Colore les points du cercle selon leur angle (`ax.scatter(..., c=angles, cmap="hsv", s=6)`) pour suivre chacun d'une étape à l'autre, mets le même cadre aux quatre axes, `ax.set_aspect("equal")` et un titre par axe. La cellule de la figure s'en sert pour $\mathbf{A}$ = `[[1.5, 1.0], [-0.5, 2.0]]`.

Dans ta copie : la deuxième étape est-elle une rotation ou une symétrie ($\det\mathbf{V}$) ? Les couleurs tournent-elles dans le même sens dans le premier et le dernier dessin, et quel lien avec le signe de $\det\mathbf{A}$ ? Vérifie que l'aire de l'ellipse finale vaut $\pi\lvert\det\mathbf{A}\rvert$.

In [ ]:
def draw_svd_steps(axes, A):
    """In the four axes: the unit circle, then its images by Vᵀ, by S Vᵀ and by U S Vᵀ = A, with the singular vectors."""
    raise NotImplementedError("np.linalg.svd(A), 200 points du cercle, puis quatre dessins")

In [ ]:
with checked("M12d.34 figure"):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
    try:
        draw_svd_steps(axes, np.array([[1.5, 1.0], [-0.5, 2.0]]))
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §3, §5 à §7. Trois erreurs à trouver dans une ACP, un test de propriétés, puis la compression d'une photo.

### Ex M12d.35 — Chasse à l'erreur : une ACP sans centrer, n et n − 1 mélangés, des axes lus en lignes au lieu de colonnes 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans une ACP écrite à la main les trois erreurs classiques, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §5 et §6 · **Parcours :** C

Un collègue a écrit cette ACP des données `X` (la cellule fournie) ; sept lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter son code**, trouve-les.

```python
n = len(X)
mu = X.mean(axis=0)
C = X.T @ X / (n - 1)                            # (A) the covariance matrix of the data
values, vectors = np.linalg.eigh(C)              # (B) the eigenvalues, in increasing order, and the eigenvectors
order = np.argsort(values)[::-1]                 # (C) the indices, from the largest eigenvalue
variances = values[order]                        # (D) the variances along the axes, in decreasing order
axes = vectors[order]                            # (E) the principal axes, in columns, in the same order
scores = (X - mu) @ axes[:, :2]                  # (F) the scores on the first two axes
check = scores.var(axis=0)                       # (G) the variances of the scores, equal to variances[:2]
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) les bonnes variances le long des trois axes (une liste, 4 décimales) ;
- c) le bon premier axe principal, avec la convention de signe (une liste, 4 décimales) ;
- d) une fois les deux autres lignes corrigées, le rapport entre ce que calcule la ligne (G) et `variances[:2]` (le même pour les deux, 4 décimales).

Pour b) et c), écris le code corrigé dans la cellule (c'est permis : seule la recherche des lignes fausses se fait sans exécuter).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Laquelle donne le résultat le plus trompeur (regarde la première variance qu'elle produit), et laquelle passerait inaperçue sans la ligne (G) ? Quel contrôle simple (fiche §6, ⚠️) aurait détecté chacune ?

In [ ]:
X = (np.random.default_rng(35).normal(size=(40, 3))
     @ np.array([[3.0, 1.0, 0.0], [0.0, 1.5, 0.5], [0.0, 0.0, 0.7]]) + np.array([20.0, -8.0, 4.0]))   # 40 examples

In [ ]:
answer_M12d_35a = ...  # a) letters, as text
answer_M12d_35b = ...  # b) a list of 3 numbers (4 decimals)
answer_M12d_35c = ...  # c) a list of 3 numbers (4 decimals)
answer_M12d_35d = ...  # d) a number (4 decimals)

In [ ]:
for letter, answer in zip("abcd", [answer_M12d_35a, answer_M12d_35b, answer_M12d_35c, answer_M12d_35d]):
    wbm.check(f"M12d.35{letter}", answer)

### Ex M12d.36 — Un test : la reconstruction de rang plein redonne X, et l'erreur décroît avec k 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriétés pour une ACP, et vérifier qu'il attrape des erreurs plausibles.  
**Prérequis :** fiche §3 et §7 ; Ex M12d.18 ; M03 (les tests) · **Parcours :** C

Un test de **propriétés** ne compare pas à des valeurs connues d'avance : il vérifie ce que tout résultat juste doit satisfaire. Écris `check_reconstruction(reconstruct, X)`, qui lève une `AssertionError` (avec `assert` ou `np.testing.assert_allclose`) si la fonction `reconstruct(X, k)`, censée rendre la reconstruction des lignes de `X` par une ACP à `k` composantes, ne vérifie pas les trois propriétés suivantes, avec $p$ le nombre de colonnes de `X`, et $e(k) = \frac{1}{n}\sum_i \lVert \mathbf{x}_i - \hat{\mathbf{x}}_i \rVert^2$ l'erreur de reconstruction moyenne :
1. **le rang plein redonne $\mathbf{X}$** : `reconstruct(X, p)` est égal à `X` à $10^{-10}$ près ;
2. **l'erreur décroît avec $k$** : $e(0) \ge e(1) \ge \cdots \ge e(p)$ (à $10^{-12}$ près) ;
3. **l'erreur est celle de la fiche** : $e(k) = \frac{n - 1}{n}\sum_{j > k}\lambda_j$ pour chaque $k$, avec $\lambda_j$ les valeurs propres de `np.cov(X, rowvar=False)` (`np.linalg.eigvalsh`, à remettre dans l'ordre décroissant), à $10^{-8}$ près relativement à la variance totale.

La cellule de vérification essaie ton test sur la bonne ACP `pca_reconstruct` (fournie) et sur trois ACP fausses : il doit accepter la première et refuser les trois autres.

Dans ta copie : laquelle des trois propriétés attrape chacune des trois ACP fausses ? Pourquoi l'ACP sans centrer passe-t-elle les propriétés 1 et 2 ? Quelle propriété ajouterais-tu pour tester des scores (moyenne nulle, variances $\lambda_j$, covariances nulles) ?

In [ ]:
X_36 = (np.random.default_rng(36).normal(size=(30, 4))
        @ np.array([[2.0, 0.3, 0.0, 0.1], [0.0, 1.2, 0.4, 0.0], [0.0, 0.0, 0.7, 0.2], [0.0, 0.0, 0.0, 0.3]])
        + np.array([5.0, -2.0, 1.0, 7.0]))


def pca_reconstruct(X, k):
    """The reconstruction of the rows of X by a PCA with k components (centered SVD)."""
    mu = X.mean(axis=0)
    _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
    return (X - mu) @ Vt[:k].T @ Vt[:k] + mu

In [ ]:
def check_reconstruction(reconstruct, X):
    """Raise AssertionError unless reconstruct(X, k) behaves as a PCA reconstruction: full rank gives X back, the mean
    error e(k) does not increase with k, and e(k) = (n − 1)/n × (sum of the forgotten eigenvalues of np.cov)."""
    raise NotImplementedError("trois propriétés, avec assert ou np.testing.assert_allclose")

In [ ]:
with checked("M12d.36 vérifications"):
    def _no_mean(X, k):                          # forgets to add the mean back
        mu = X.mean(axis=0)
        _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
        return (X - mu) @ Vt[:k].T @ Vt[:k]

    def _increasing(X, k):                       # the axes of the SMALLEST variances first
        mu = X.mean(axis=0)
        _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
        W = Vt[::-1][:k]
        return (X - mu) @ W.T @ W + mu

    def _not_centered(X, k):                     # the SVD of X itself
        _, _, Vt = np.linalg.svd(X, full_matrices=False)
        return X @ Vt[:k].T @ Vt[:k]

    results_36 = []
    for name, function, should_pass in (("la bonne ACP, pca_reconstruct", pca_reconstruct, True),
                                        ("une ACP qui oublie de rajouter la moyenne", _no_mean, False),
                                        ("une ACP qui prend d'abord les axes de PLUS PETITE variance", _increasing, False),
                                        ("une ACP sans centrer", _not_centered, False)):
        try:
            check_reconstruction(function, X_36.copy())
            passed = True
        except AssertionError:
            passed = False
        results_36.append(passed == should_pass)
        verdict = ("✅" if passed == should_pass else "❌") + (" ton test accepte" if passed else " ton test refuse")
        print(f"{verdict} {name}")
    if all(results_36):
        print("✅ Ex M12d.36 : ton test accepte la bonne ACP et attrape les trois erreurs.")
    else:
        print("❌ Ex M12d.36 : ton test doit accepter la bonne ACP et refuser (AssertionError) les trois autres : "
              "relis les trois propriétés de l'énoncé.")

### Ex M12d.37 — Défi : compresser une image centrée en gardant 93 % de son énergie avec au plus 5 % des nombres 🏆 ★★★ ⏱️ 30 min
**Objectif :** trouver une représentation de rang faible d'une photo plus économe que la SVD de l'image entière.  
**Prérequis :** fiche §3 et §6 ; Ex M12d.31 et M12d.32 · **Parcours :** C

La photo de la tasse (`IMAGE_37`, $400 \times 600$ niveaux de gris) a $240\,000$ pixels. **L'objectif** : écrire
- `compress(A)`, qui rend une **liste de tableaux NumPy** (la représentation compressée), et
- `decompress(parts, shape)`, qui reconstruit à partir de cette liste seule une image $\hat{\mathbf{A}}$ de forme `shape`,

de sorte que l'**énergie gardée** $1 - \lVert \mathbf{A} - \hat{\mathbf{A}} \rVert_F^2 / \lVert \mathbf{A} - \bar{a} \rVert_F^2$ (avec $\bar{a}$ le niveau de gris moyen : l'énergie de l'image **centrée**) soit d'au moins 93 %, avec au plus 5 % des nombres : la somme des tailles (`.size`) des tableaux rendus par `compress` ne doit pas dépasser `BUDGET_37` $= 12\,000$. La forme `shape` et les constantes écrites dans ton code (une taille, un rang) ne comptent pas ; le niveau moyen, s'il est stocké, compte pour un nombre. La vérification essaie aussi la photo retournée de gauche à droite (une compression honnête y fait aussi bien). NumPy est permis, `np.linalg.svd` et ta librairie compris ; les transformées de Fourier ou en cosinus (`np.fft`, `scipy`), scikit-learn et les bibliothèques d'images ou de compression sont interdits.

Point de départ (M12d.32) : la SVD de l'image entière centrée, au plus grand rang qui tient dans le budget ($k(m + n + 1) + 1 \le 12\,000$, soit $k = 11$), ne garde qu'environ 89 % de l'énergie. Il faut une **autre matrice** que l'image elle-même, dont les valeurs singulières décroissent plus vite. Une dernière cellule enregistre si l'objectif est atteint.

Dans ta copie : quelle matrice as-tu décomposée, et pourquoi ses valeurs singulières décroissent-elles plus vite que celles de l'image ? Comment as-tu choisi les réglages (une taille, un rang), et que gardes-tu au mieux ? Regarde ta reconstruction : quels défauts voit-on (des blocs ?), que la norme de Frobenius ne mesure pas ?

In [ ]:
IMAGE_37 = wbm.datasets.load_image("tasse")       # 400 × 600 grey levels, CC0
BUDGET_37 = int(0.05 * IMAGE_37.size)               # at most 12 000 numbers

In [ ]:
def compress(A):
    """A list of NumPy arrays from which decompress(parts, A.shape) rebuilds an approximation of A."""
    raise NotImplementedError("une liste de tableaux : au plus BUDGET_37 nombres en tout")


def decompress(parts, shape):
    """The approximation of the image of the given shape, from the list of arrays returned by compress."""
    raise NotImplementedError("reconstruis l'image à partir de parts seulement")

In [ ]:
with checked("M12d.37 défi"):
    import inspect
    import re
    problems, ok_37, ran_37 = [], False, False
    try:
        source_37 = inspect.getsource(compress) + inspect.getsource(decompress)
    except (OSError, TypeError):
        source_37 = ""
    banned_37 = sorted(set(re.findall(r"\b(fft|dct|scipy|sklearn|PIL|imageio|cv2|zlib|pickle|bz2|lzma|gzip|load_image|"
                                      r"IMAGE_37)\b", source_37)))
    if banned_37:
        ran_37 = True
        problems.append(f"compress ou decompress utilise {', '.join(banned_37)}, interdit dans ce défi")
    images_37 = [("la photo", IMAGE_37), ("la photo retournée de gauche à droite", IMAGE_37[:, ::-1].copy())]
    for name_t, A_t in (images_37 if not banned_37 else ()):
        try:
            parts_t = list(compress(A_t.copy()))
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            ran_37 = True
            problems.append(f"compress lève {type(exc).__name__} ({exc})")
            break
        ran_37 = True
        arrays_t = [np.asarray(part) for part in parts_t]
        if not arrays_t or not all(np.issubdtype(a.dtype, np.integer) or np.issubdtype(a.dtype, np.floating) for a in arrays_t):
            problems.append("compress doit rendre une liste de tableaux NumPy de nombres réels")
            break
        count_t = sum(a.size for a in arrays_t)
        try:
            B_t = np.asarray(decompress([a.copy() for a in arrays_t], A_t.shape), dtype=float)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"decompress lève {type(exc).__name__} ({exc})")
            break
        if B_t.shape != A_t.shape:
            problems.append(f"decompress rend la forme {B_t.shape}, pas {A_t.shape}")
            break
        kept_t = 1 - float(np.sum((A_t - B_t) ** 2) / np.sum((A_t - A_t.mean()) ** 2))
        print(f"{name_t} : {count_t} nombres ({count_t / A_t.size:.2%}), énergie gardée {kept_t:.2%}".replace(".", ","))
        if count_t > BUDGET_37:
            problems.append(f"{name_t} : {count_t} nombres stockés, au-dessus du budget de {BUDGET_37}")
        if not kept_t >= 0.93:
            problems.append(f"{name_t} : {kept_t:.2%} de l'énergie gardée, sous les 93 %".replace(".", ","))
        if problems:
            break
    if problems:
        print("❌ Ex M12d.37 : " + " ; ".join(problems) + ".")
    elif ran_37:
        ok_37 = True
        print("✅ Ex M12d.37 : défi réussi : au moins 93 % de l'énergie avec au plus 5 % des nombres, sur les deux images.")
with checked("M12d.37"):
    if not ran_37:                               # ⏳ while compress is not written
        raise NotImplementedError("compress, ou une fonction qu'elle appelle, n'est pas encore écrite")
    wbm.check("M12d.37", bool(ok_37), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la SVD d'une petite matrice par AᵀA, et l'interpréter comme rotation, étirement, rotation ?
2. Sais-tu calculer une approximation de rang faible, son erreur et son coût de stockage ?
3. Sais-tu calculer une pseudo-inverse et un conditionnement, et dire ce qu'ils signifient ?
4. Sais-tu faire une ACP à la main (covariance, axes, scores, reconstruction) et lire une variance expliquée ?
5. Sais-tu dire ce que t-SNE et UMAP gardent, et ce qu'une de leurs cartes ne permet pas de conclure ?

**La suite** : le chapitre 12 de workbookIA (préparation des données), puis M13a, vraisemblance et régression logistique.